# IranSeda preparation pipeline — saved-artifact demo

This notebook presents the core IranSeda dataset-preparation stages from their saved JSON, JSONL, and TSV artifacts. It **does not load or run VAD, Whisper, or an LLM**. Random examples are reproducible, and audio playback is optional.

**Saved flow:** segmentation and VAD → Whisper transcription and deterministic Persian normalization → contextual LLM refinement and validation → final refined labels.

## Configuration

Set one pipeline root for the normal layout. Any non-`None` entry in `ARTIFACT_OVERRIDES` takes priority and may point to an artifact from another directory. Relative paths are resolved from the repository root. `ARTIFACT_RECORD_LIMITS` places an independent hard cap on the number of records read from every JSONL or TSV artifact.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# The environment override is useful on a data server and in automated checks.
PIPELINE_ROOT = Path(os.environ.get('IRANSEDA_PIPELINE_ROOT', 'data/iranseda/segmented/flac-v1'))

# Explicit paths win over PIPELINE_ROOT / <standard filename>.
ARTIFACT_OVERRIDES: dict[str, str | Path | None] = {
    'segmentation_summary': None,
    'segments': None,
    'vad_intervals': None,
    'transcription_summary': None,
    'transcriptions': None,
    'transcription_rejected': None,
    'refinement_summary': None,
    'refinements': None,
    'refinement_rejected': None,
    'refined_transcription': None,
}

# Hard per-file caps keep large line-oriented artifacts from exhausting notebook memory.
ARTIFACT_RECORD_LIMITS: dict[str, int] = {
    'segments': 10_000,
    'vad_intervals': 10_000,
    'transcriptions': 10_000,
    'transcription_rejected': 10_000,
    'refinements': 10_000,
    'refinement_rejected': 10_000,
    'refined_transcription': 10_000,
}

RANDOM_SEED = 1337
SEGMENT_EXAMPLE_COUNT = 3
TRANSCRIPTION_EXAMPLE_COUNT = 6
LLM_GALLERY_COUNT = 8
PLAY_AUDIO = os.environ.get('IRANSEDA_DEMO_AUDIO', 'true').lower() not in {'0', 'false', 'no'}

In [ ]:
import csv
import html
import json
import random
import statistics
from collections import Counter
from collections.abc import Iterable, Sequence
from typing import Any

from IPython.display import Audio, HTML, Markdown, display

ARTIFACT_FILENAMES = {
    'segmentation_summary': 'summary.json',
    'segments': 'segments.jsonl',
    'vad_intervals': 'vad_intervals.jsonl',
    'transcription_summary': 'transcription_summary.json',
    'transcriptions': 'transcriptions.jsonl',
    'transcription_rejected': 'transcription_rejected.jsonl',
    'refinement_summary': 'refinement_summary.json',
    'refinements': 'refinements.jsonl',
    'refinement_rejected': 'refinement_rejected.jsonl',
    'refined_transcription': 'refined_transcription.tsv',
}


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPO_ROOT = find_repo_root(Path.cwd())


def absolute_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    return candidate.resolve() if candidate.is_absolute() else (REPO_ROOT / candidate).resolve()


def resolve_artifact_paths(
    root: str | Path, overrides: dict[str, str | Path | None]
) -> dict[str, Path]:
    root_path = absolute_path(root)
    unknown = set(overrides) - set(ARTIFACT_FILENAMES)
    if unknown:
        raise KeyError(f'Unknown artifact override(s): {sorted(unknown)}')
    return {
        name: absolute_path(overrides[name]) if overrides.get(name) is not None else root_path / filename
        for name, filename in ARTIFACT_FILENAMES.items()
    }


def read_json(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(value, dict):
        raise ValueError(f'Expected a JSON object in {path}')
    return value


def read_jsonl(path: Path, limit: int) -> list[dict[str, Any]]:
    if limit < 1:
        raise ValueError('JSONL record limit must be at least 1.')
    rows: list[dict[str, Any]] = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise ValueError(f'Expected an object at {path}:{line_number}')
            rows.append(value)
            if len(rows) >= limit:
                break
    return rows


def read_tsv(path: Path, limit: int) -> list[dict[str, str]]:
    if limit < 1:
        raise ValueError('TSV record limit must be at least 1.')
    with path.open(encoding='utf-8', newline='') as handle:
        rows: list[dict[str, str]] = []
        for row in csv.DictReader(handle, delimiter='\t'):
            rows.append(row)
            if len(rows) >= limit:
                break
        return rows


def optional_load(path: Path, reader: Any, default: Any) -> Any:
    if not path.is_file():
        print(f'Optional artifact not found; its view will be skipped: {path}')
        return default
    return reader(path)


def require_files(paths: dict[str, Path], names: Sequence[str]) -> None:
    missing = [f'{name}: {paths[name]}' for name in names if not paths[name].is_file()]
    if missing:
        raise FileNotFoundError(
            'Required IranSeda artifacts are missing. Set PIPELINE_ROOT or an explicit override:\n'
            + '\n'.join(missing)
        )


def sample_rows(rows: Sequence[dict[str, Any]], count: int, seed: int) -> list[dict[str, Any]]:
    if count < 0:
        raise ValueError('Sample count cannot be negative.')
    return random.Random(seed).sample(list(rows), min(count, len(rows)))


def mixed_llm_sample(
    accepted: Sequence[dict[str, Any]], rejected: Sequence[dict[str, Any]], count: int, seed: int
) -> list[dict[str, Any]]:
    if count < 0:
        raise ValueError('Sample count cannot be negative.')
    rng = random.Random(seed)
    accepted_rows = list(accepted)
    rejected_rows = list(rejected)
    rng.shuffle(accepted_rows)
    rng.shuffle(rejected_rows)
    if accepted_rows and rejected_rows:
        accepted_count = min(len(accepted_rows), (count + 1) // 2)
        rejected_count = min(len(rejected_rows), count - accepted_count)
        remaining = count - accepted_count - rejected_count
        accepted_count += min(len(accepted_rows) - accepted_count, remaining)
        remaining = count - accepted_count - rejected_count
        rejected_count += min(len(rejected_rows) - rejected_count, remaining)
        selected = accepted_rows[:accepted_count] + rejected_rows[:rejected_count]
    else:
        selected = (accepted_rows or rejected_rows)[:count]
    rng.shuffle(selected)
    return selected


def by_id(rows: Iterable[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    return {str(row['id']): row for row in rows if row.get('id') is not None}


def fmt(value: Any, digits: int = 3) -> str:
    return f'{value:.{digits}f}' if isinstance(value, float) else str(value)


def show_mapping(title: str, values: dict[str, Any]) -> None:
    rows = ''.join(
        f'<tr><th>{html.escape(str(key))}</th><td>{html.escape(fmt(value))}</td></tr>'
        for key, value in values.items()
    )
    display(HTML(f'<h4>{html.escape(title)}</h4><table>{rows}</table>'))


def show_text(label: str, value: Any, *, rtl: bool = True) -> None:
    direction = 'rtl' if rtl else 'ltr'
    display(HTML(
        f'<strong>{html.escape(label)}</strong>'
        f'<pre dir="{direction}" style="white-space:pre-wrap">{html.escape(str(value))}</pre>'
    ))


def clip_path(record: dict[str, Any], root: Path) -> Path | None:
    raw = record.get('path')
    if not isinstance(raw, str):
        return None
    candidate = Path(raw)
    if candidate.is_absolute():
        return candidate if candidate.is_file() else None
    resolved = (root / candidate).resolve()
    return resolved if resolved.is_file() else None


def maybe_play_audio(record: dict[str, Any], root: Path) -> None:
    if not PLAY_AUDIO:
        return
    path = clip_path(record, root)
    if path is not None:
        display(Audio(filename=str(path)))
    else:
        print(f"Audio unavailable for {record.get('path', record.get('id', 'record'))}; continuing with metadata.")

In [ ]:
artifact_paths = resolve_artifact_paths(PIPELINE_ROOT, ARTIFACT_OVERRIDES)
require_files(artifact_paths, ('segments', 'transcriptions'))
if not artifact_paths['refinements'].is_file() and not artifact_paths['refinement_rejected'].is_file():
    raise FileNotFoundError(
        'At least one LLM audit artifact is required: refinements.jsonl or refinement_rejected.jsonl.'
    )

pipeline_root = absolute_path(PIPELINE_ROOT)
audio_root = artifact_paths['segments'].parent
segments = read_jsonl(artifact_paths['segments'], ARTIFACT_RECORD_LIMITS['segments'])
vad_intervals = optional_load(artifact_paths['vad_intervals'], lambda path: read_jsonl(path, ARTIFACT_RECORD_LIMITS['vad_intervals']), [])
segmentation_summary = optional_load(artifact_paths['segmentation_summary'], read_json, {})
transcriptions = read_jsonl(artifact_paths['transcriptions'], ARTIFACT_RECORD_LIMITS['transcriptions'])
transcription_rejected = optional_load(artifact_paths['transcription_rejected'], lambda path: read_jsonl(path, ARTIFACT_RECORD_LIMITS['transcription_rejected']), [])
transcription_summary = optional_load(artifact_paths['transcription_summary'], read_json, {})
refinements = optional_load(artifact_paths['refinements'], lambda path: read_jsonl(path, ARTIFACT_RECORD_LIMITS['refinements']), [])
refinement_rejected = optional_load(artifact_paths['refinement_rejected'], lambda path: read_jsonl(path, ARTIFACT_RECORD_LIMITS['refinement_rejected']), [])
refinement_summary = optional_load(artifact_paths['refinement_summary'], read_json, {})
refined_rows = optional_load(artifact_paths['refined_transcription'], lambda path: read_tsv(path, ARTIFACT_RECORD_LIMITS['refined_transcription']), [])
if not refinements and not refinement_rejected:
    raise ValueError('The LLM audit artifacts contain no accepted or rejected records to demonstrate.')

print(f'Pipeline root: {pipeline_root}')
for name, path in artifact_paths.items():
    state = 'found' if path.is_file() else 'not found (optional)'
    print(f'{name:26} {state:20} {path}')

## 1. Segmentation and VAD outputs

The segmenter converts long recordings into deterministic, non-overlapping clips. Its manifests retain boundaries, speech coverage, checksums, and the VAD audit trail.

In [ ]:
durations = [float(row['duration_sec']) for row in segments if isinstance(row.get('duration_sec'), (int, float))]
speech_ratios = [float(row['speech_ratio']) for row in segments if isinstance(row.get('speech_ratio'), (int, float))]
boundary_counts = Counter(str(row.get('boundary_type', 'unknown')) for row in segments)
summary = {
    'segments': len(segments),
    'sources': len({row.get('source_id') for row in segments if row.get('source_id') is not None}),
    'total audio hours': round(sum(durations) / 3600, 3),
    'mean segment seconds': round(statistics.mean(durations), 3) if durations else 'n/a',
    'mean speech ratio': round(statistics.mean(speech_ratios), 3) if speech_ratios else 'n/a',
    'VAD intervals': len(vad_intervals),
}
summary.update({f'boundary: {name}': count for name, count in boundary_counts.most_common()})
show_mapping('Segmentation overview', {**segmentation_summary, **summary})

In [ ]:
for index, segment in enumerate(sample_rows(segments, SEGMENT_EXAMPLE_COUNT, RANDOM_SEED + 1), start=1):
    display(Markdown(f"### Segment {index}: `{segment.get('id', 'unknown')}`"))
    show_mapping('Saved segment metadata', {
        key: segment.get(key) for key in (
            'source_id', 'path', 'start_sec', 'end_sec', 'duration_sec',
            'speech_seconds', 'speech_ratio', 'boundary_type',
            'boundary_silence_sec', 'energy_dip_db', 'clip_checksum', 'config_digest',
        )
    })
    maybe_play_audio(segment, audio_root)

## 2. Whisper and deterministic normalization outputs

These are persisted inference results. The notebook compares Whisper's raw text with the deterministic Persian-normalized label; it performs no transcription itself.

In [ ]:
show_mapping('Transcription overview', {
    **transcription_summary,
    'accepted manifest rows': len(transcriptions),
    'rejected manifest rows': len(transcription_rejected),
})
segment_index = by_id(segments)
for index, row in enumerate(sample_rows(transcriptions, TRANSCRIPTION_EXAMPLE_COUNT, RANDOM_SEED + 2), start=1):
    display(Markdown(f"### Transcript {index}: `{row.get('id', 'unknown')}`"))
    show_text('Raw Whisper output', row.get('raw_transcript', '(not recorded)'))
    show_text('Deterministically normalized text', row.get('normalized_transcript', '(not recorded)'))
    show_mapping('Inference provenance', {
        'source_id': row.get('source_id'),
        'path': row.get('path'),
        'generation': row.get('generation'),
        'config_digest': row.get('config_digest'),
    })
    maybe_play_audio(segment_index.get(str(row.get('id')), row), audio_root)

## 3. Exact contextual LLM audit

The following example is sampled from the saved accepted and rejected audit records. `rendered_prompt` is displayed verbatim: it is the exact persisted LLM input, not a prompt reconstructed by this notebook.

In [ ]:
all_llm_records = [*refinements, *refinement_rejected]
audit_record = sample_rows(all_llm_records, 1, RANDOM_SEED + 3)[0]
status = 'rejected' if audit_record.get('reason') else 'accepted'
display(Markdown(f"### `{audit_record.get('id', 'unknown')}` — **{status}**"))
show_text('Audiobook title', audit_record.get('title') or '(omitted)')
show_text('Audiobook description', audit_record.get('description') or '(omitted)')
show_text('Refined preceding context', json.dumps(audit_record.get('preceding_context', []), ensure_ascii=False, indent=2))
show_text('Target Whisper text', audit_record.get('target_whisper_text', audit_record.get('target', {}).get('text', '')))
show_text('Following Whisper context', json.dumps(audit_record.get('following_context', []), ensure_ascii=False, indent=2))
show_text('Exact rendered prompt', audit_record.get('rendered_prompt', '(not recorded)'))
show_mapping('Model and contract', {
    'model': audit_record.get('model'),
    'model_parameters': audit_record.get('model_parameters'),
    'prompt_version': audit_record.get('prompt_version'),
    'schema_version': audit_record.get('schema_version'),
    'response_schema': audit_record.get('response_schema'),
})
show_text('Raw LLM response text', audit_record.get('raw_response_text', audit_record.get('raw_response')))
show_text('Parsed response', json.dumps(audit_record.get('parsed_response'), ensure_ascii=False, indent=2))
show_mapping('Validation result', {
    'status': status,
    'reason': audit_record.get('reason'),
    'detail': audit_record.get('detail'),
    'validation_metrics': audit_record.get('validation_metrics'),
})

## 4. Multiple LLM inputs and outputs

The gallery balances accepted and rejected records as far as the available artifacts permit.

In [ ]:
gallery = mixed_llm_sample(refinements, refinement_rejected, LLM_GALLERY_COUNT, RANDOM_SEED + 4)
for index, row in enumerate(gallery, start=1):
    status = 'rejected' if row.get('reason') else 'accepted'
    parsed = row.get('parsed_response')
    cleaned = row.get('cleaned_text')
    if cleaned is None and isinstance(parsed, dict):
        cleaned = parsed.get('cleaned_text')
    display(Markdown(f"### {index}. `{row.get('id', 'unknown')}` — **{status}**"))
    show_text('LLM input target', row.get('target_whisper_text', row.get('target', {}).get('text', '')))
    show_text('LLM output', cleaned if cleaned is not None else row.get('raw_response_text', '(no text output)'))
    metrics = row.get('validation_metrics') if isinstance(row.get('validation_metrics'), dict) else {}
    show_mapping('Decision', {
        'uncertain': parsed.get('uncertain') if isinstance(parsed, dict) else None,
        'normalized edit distance': metrics.get('normalized_edit_distance'),
        'rejection reason': row.get('reason'),
    })

## 5. Refinement decisions and final labels

In [ ]:
edit_distances = [
    float(row['validation_metrics']['normalized_edit_distance'])
    for row in all_llm_records
    if isinstance(row.get('validation_metrics'), dict)
    and isinstance(row['validation_metrics'].get('normalized_edit_distance'), (int, float))
]
rejection_reasons = Counter(str(row.get('reason', 'unknown')) for row in refinement_rejected)
final_summary = {
    **refinement_summary,
    'accepted audit rows': len(refinements),
    'rejected audit rows': len(refinement_rejected),
    'final TSV rows': len(refined_rows),
    'mean normalized edit distance': round(statistics.mean(edit_distances), 4) if edit_distances else 'n/a',
}
final_summary.update({f'rejection: {reason}': count for reason, count in rejection_reasons.most_common()})
show_mapping('Refinement overview', final_summary)

display(Markdown('### Final `refined_transcription.tsv` preview'))
for index, row in enumerate(refined_rows[:10], start=1):
    display(HTML(
        f'<p><strong>{index}. {html.escape(row.get("path", ""))}</strong><br>'
        f'<span dir="rtl">{html.escape(row.get("sentence", ""))}</span></p>'
    ))

---
Every value above comes from a saved pipeline artifact. Change the seed to inspect another reproducible sample, or provide explicit artifact overrides to compare outputs from different runs.